# Tidy up FITS headers for publication
## Oct 21 version just for making files to share with GMIMS consortium. Will need to be updated.
## A. Ordog Oct 21
### Files were copied and modified from /srv/data/cgps-gmims/data_for_paper/conv_regrid/
### Currently differnt functions used for PI and peak FD versus RM maps because the latter have extensions


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u

In [ ]:
hdu = fits.open('/srv/data/cgps-gmims/conv_regrid/PA_A_C_conv4_regrd.fits')
data = hdu[0].data

plt.hist(data.flatten(),bins=201);

In [ ]:
plt.ax = plt.subplots(1,1,figsize=(20,6))

plt.imshow(data,vmin=-np.pi,vmax=np.pi,cmap='twilight')

## Plotting function

In [ ]:
def plot_3panel_maps(map1,map2,map3,wcs,labels,title,
                     lrange=[180,60],brange=[-5,5],vmin=0,vmax=1,cmap='viridis'):

    width = 20
    height = 3.2*width*(brange[1]-brange[0])/(lrange[0]-lrange[1])
    print(height)
    
    fig = plt.figure(figsize=(width,height))
    fs = 10

    #plt.subplots_adjust(hspace=0.0,left=0.05,right=0.999,top=0.95,bottom=0.05,wspace=0)

    crange = SkyCoord(lrange, brange, frame=Galactic, unit=(u.deg, u.deg))

    ax1 = fig.add_subplot(311, projection=wcs.celestial)
    ax2 = fig.add_subplot(312, projection=wcs.celestial)
    ax3 = fig.add_subplot(313, projection=wcs.celestial)

    im1 = ax1.imshow(map1,vmin=vmin,vmax=vmax,cmap=cmap)
    im2 = ax2.imshow(map2,vmin=vmin,vmax=vmax,cmap=cmap)
    im3 = ax3.imshow(map3,vmin=vmin,vmax=vmax,cmap=cmap)

    axs = [ax1,ax2,ax3]
    ims = [im1,im2,im3]

    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.82, pad=0.005, aspect=12)
        cbar.set_label(label=labels[i],fontsize=fs)
        cbar.ax.tick_params(labelsize=fs) 
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('Latitude',fontsize=fs)
        axs[i].tick_params(axis='both',labelsize=fs)
    
    ax3.set_xlabel('Longitude',fontsize=fs)

    ax1.set_title(title,fontsize=fs+2)

    return

## Read in PI data

In [ ]:
hdu_C_PI  = fits.open('/srv/data/cgps-gmims/data_for_paper/PI_C_conv4_regrd_PI_of_mean.fits')
hdu_G_PI  = fits.open('/srv/data/cgps-gmims/data_for_paper/PI_G_regrd_PI_of_mean.fits')
hdu_CG_PI = fits.open('/srv/data/cgps-gmims/data_for_paper/PI_CG_conv4_regrd_PI_of_mean.fits')

C_PI  = hdu_C_PI[0].data
G_PI  = hdu_G_PI[0].data
CG_PI = hdu_CG_PI[0].data

print(C_PI.shape)
print(G_PI.shape)
print(CG_PI.shape)

hdr_C_PI  = hdu_C_PI[0].header
hdr_G_PI  = hdu_G_PI[0].header
hdr_CG_PI = hdu_CG_PI[0].header


In [ ]:
print(repr(hdr_CG_PI))


In [ ]:
plot_3panel_maps(C_PI, G_PI, CG_PI, WCS(hdr_C_PI), ['PI (K)','PI (K)','PI (K)'],'PI',
                 lrange=[195,50], brange=[-5,7], vmin=0, vmax=0.5, cmap='viridis')

## Read in RM data

In [ ]:
hdu_C_RM  = fits.open('/srv/data/cgps-gmims/data_for_paper/RM_C_conv4_regrd.fits')
hdu_G_RM  = fits.open('/srv/data/cgps-gmims/data_for_paper/RM_G_regrd.fits')
hdu_CG_RM = fits.open('/srv/data/cgps-gmims/data_for_paper/RM_CG_conv4_regrd.fits')

ext = 0

C_RM  = hdu_C_RM[ext].data
G_RM  = hdu_G_RM[ext].data
CG_RM = hdu_CG_RM[ext].data

print(C_RM.shape)
print(G_RM.shape)
print(CG_RM.shape)

hdr_C_RM  = hdu_C_RM[ext].header
hdr_G_RM  = hdu_G_RM[ext].header
hdr_CG_RM = hdu_CG_RM[ext].header

In [ ]:
print(repr(hdr_C_RM))


In [ ]:
plot_3panel_maps(C_RM, G_RM, CG_RM, WCS(hdr_C_RM), ['RM (rad/m^2)','RM (rad/m^2)','RM (rad/m^2)'],'RM',
                 lrange=[195,50], brange=[-5,7], vmin=-200, vmax=200, cmap='RdBu_r')

## Read in GMIMS peak FD

In [ ]:
hdu_FD  = fits.open('/srv/data/cgps-gmims/gmims_FD/phi_peak_regrd.fits')
hdr_FD  = hdu_FD[0].header
peak_FD = hdu_FD[0].data

print(peak_FD.shape)

print(repr(hdr_FD))

In [ ]:
plot_3panel_maps(C_RM, G_RM, peak_FD, WCS(hdr_C_RM), ['RM (rad/m^2)','RM (rad/m^2)','RM (rad/m^2)'],'RM',
                 lrange=[195,50], brange=[-5,7], vmin=-200, vmax=200, cmap='RdBu_r')

## Modify the headers

In [ ]:
def modify_header_and_save_file(hdr, data, telescope, units, comment1, comment2, comment3, outfile):

    hdr_new = hdr.copy()

    hdr_new['BITPIX']  = -32
    hdr_new['BUNIT'] = units
    
    # delete 'object'
    del hdr_new['OBJECT']

    # include telescope
    hdr_new['TELESCOP'] = telescope

    # add notes in 'comment'
    hdr_new['COMMENT'] = comment1
    hdr_new['COMMENT'] = comment2
    hdr_new['COMMENT'] = comment3

    print(repr(hdr_new))
    print('')

    fits.writeto(outfile, data.astype('float32'), hdr_new, overwrite=True)
    
    return

In [ ]:
data_list = [C_PI, G_PI, CG_PI, peak_FD]

unit_list = ['K   ', 'K   ', 'K   ', 'RAD/M^2   ']

hdr_list  = [hdr_C_PI, hdr_G_PI, hdr_CG_PI, hdr_FD]

telescope_list = ['DRAO ST', 'John A. Galt 26-m', 'DRAO ST + Galt 26-m', 'John A. Galt 26-m']

outdir = '/srv/data/cgps-gmims/data_for_paper/'

outfile_list = [outdir+'aperture_synthesis_PI.fits', outdir+'single_antenna_PI.fits', outdir+'combined_PI.fits',
                outdir+'single_antenna_peakFD.fits']

comment1_list = ['    PI from DRAO ST CGPS Q & U files convolved to 3 arcmin',
                 '    PI from GMIMS-HBN Q & U files reprojected to AS grid',
                 '    PI from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin',
                 '    fitted peak Faraday depth from full GMIMS-HBN reprojected to AS grid']

comment2_list = [' ', ' ', ' ', ' ']

comment3_list = [' ', ' ', ' ', ' ']

In [ ]:
for i in range(3,len(data_list)):

    modify_header_and_save_file(hdr_list[i], data_list[i], telescope_list[i], unit_list[i],
                                comment1_list[i], comment2_list[i], comment3_list[i], outfile_list[i])

In [ ]:
def modify_hdu_and_save_file(hdu, telescope, units, comment1, comment2, comment3, outfile):

    for j in range(0,6):
    
        hdu[j].header['BITPIX']  = -32
        hdu[j].header['BUNIT'] = units[j]
        
        # delete 'object'
        del hdu[j].header['OBJECT']
    
        # include telescope
        hdu[j].header['TELESCOP'] = telescope
    
        # add notes in 'comment'
        hdu[j].header['COMMENT'] = comment1
        hdu[j].header['COMMENT'] = comment2
        hdu[j].header['COMMENT'] = comment3

        hdu[j].data = hdu[j].data.astype(np.float32)

    #print(repr(hdr_new))
    #print('')

    #fits.writeto(outfile, data.astype('float32'), hdr_new, overwrite=True)
    hdu.writeto(outfile,overwrite=True)
    
    return

In [ ]:
def modify_hdu_and_save_file_v2(hdu, telescope, units, comment1, comment2, comment3, outfile):

    # Note: _v2 has only 4 extensions instead of 6
    for j in range(0,4):
    
        hdu[j].header['BITPIX']  = -32
        hdu[j].header['BUNIT'] = units[j]
        
        # delete 'object'
        del hdu[j].header['OBJECT']
    
        # include telescope
        hdu[j].header['TELESCOP'] = telescope
    
        # add notes in 'comment'
        hdu[j].header['COMMENT'] = comment1
        hdu[j].header['COMMENT'] = comment2
        hdu[j].header['COMMENT'] = comment3

        hdu[j].data = hdu[j].data.astype(np.float32)

    #print(repr(hdr_new))
    #print('')

    #fits.writeto(outfile, data.astype('float32'), hdr_new, overwrite=True)
    hdu.writeto(outfile,overwrite=True)
    
    return

In [ ]:
hdu_list = [hdu_C_RM, hdu_G_RM, hdu_CG_RM]

telescope_list = ['DRAO ST', 'John A. Galt 26-m', 'DRAO ST + Galt 26-m']

unit_list = ['RAD/M^2', 'RAD', ' ', ' ', 'RAD/M^2', 'RAD']

outdir = '/srv/data/cgps-gmims/data_for_paper/'

outfile_list = [outdir+'aperture_synthesis_RM.fits', outdir+'single_antenna_RM.fits', outdir+'combined_RM.fits']

comment1_list = ['    RM from DRAO ST CGPS Q & U files convolved to 3 arcmin',
                 '    RM from GMIMS-HBN Q & U files reprojected to AS grid',
                 '    RM from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin']

comment2_list = ['    Extensions: 1 linear fit RM, 2 polarization angle y-intercept,',
                 '    Extensions: 1 linear fit RM, 2 polarization angle y-intercept,',
                 '    Extensions: 1 linear fit RM, 2 polarization angle y-intercept,']

comment3_list = ['    3 r-value, 4 p-value, 5 linear fit error, 6 PA y-intercept error',
                 '    3 r-value, 4 p-value, 5 linear fit error, 6 PA y-intercept error',
                 '    3 r-value, 4 p-value, 5 linear fit error, 6 PA y-intercept error']

In [ ]:
for i in range(0,len(hdu_list)):

    modify_hdu_and_save_file(hdu_list[i], telescope_list[i], unit_list,
                                comment1_list[i], comment2_list[i], comment3_list[i], outfile_list[i])

## PA and Stokes Q/U files for referee

In [ ]:
hdu_CG_PA_A  = fits.open('/srv/data/cgps-gmims/conv_regrid/PA_A_CG_conv4_regrd.fits')
hdu_CG_PA_B  = fits.open('/srv/data/cgps-gmims/conv_regrid/PA_B_CG_conv4_regrd.fits')
hdu_CG_PA_C  = fits.open('/srv/data/cgps-gmims/conv_regrid/PA_C_CG_conv4_regrd.fits')
hdu_CG_PA_D  = fits.open('/srv/data/cgps-gmims/conv_regrid/PA_D_CG_conv4_regrd.fits')

CG_PA_A = hdu_CG_PA_A[0].data
CG_PA_B = hdu_CG_PA_B[0].data
CG_PA_C = hdu_CG_PA_C[0].data
CG_PA_D = hdu_CG_PA_D[0].data

print(CG_PA_A.shape)

hdr_CG_PA_A = hdu_CG_PA_A[0].header
hdr_CG_PA_B = hdu_CG_PA_B[0].header
hdr_CG_PA_C = hdu_CG_PA_C[0].header
hdr_CG_PA_D = hdu_CG_PA_D[0].header


In [ ]:
data_list = [CG_PA_A, CG_PA_B, CG_PA_C, CG_PA_D]

unit_list = ['RAD   ', 'RAD   ', 'RAD   ', 'RAD   ']

hdr_list  = [hdr_CG_PA_A, hdr_CG_PA_B, hdr_CG_PA_C, hdr_CG_PA_D]

telescope_list = ['DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m']

outdir = '/srv/data/cgps-gmims/data_for_paper/'

outfile_list = [outdir+'combined_PA_chanA.fits', outdir+'combined_PA_chanB.fits',
                outdir+'combined_PA_chanC.fits', outdir+'combined_PA_chanD.fits']

comment1_list = ['    PA from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin',
                 '    PA from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin',
                 '    PA from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin',
                 '    PA from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin']

comment2_list = ['    Channel A, 1407 MHz',
                 '    Channel B, 1414 MHz',
                 '    Channel C, 1428 MHz',
                 '    Channel D, 1435 MHz',]

comment3_list = [' ', ' ', ' ', ' ']

In [ ]:
for i in range(0,len(data_list)):

    modify_header_and_save_file(hdr_list[i], data_list[i], telescope_list[i], unit_list[i],
                                comment1_list[i], comment2_list[i], comment3_list[i], outfile_list[i])

In [ ]:
hdu_CG_QA  = fits.open('/srv/data/cgps-gmims/conv_regrid/QA_CG_conv4_regrd.fits')
hdu_CG_QB  = fits.open('/srv/data/cgps-gmims/conv_regrid/QB_CG_conv4_regrd.fits')
hdu_CG_QC  = fits.open('/srv/data/cgps-gmims/conv_regrid/QC_CG_conv4_regrd.fits')
hdu_CG_QD  = fits.open('/srv/data/cgps-gmims/conv_regrid/QD_CG_conv4_regrd.fits')

hdu_CG_UA  = fits.open('/srv/data/cgps-gmims/conv_regrid/UA_CG_conv4_regrd.fits')
hdu_CG_UB  = fits.open('/srv/data/cgps-gmims/conv_regrid/UB_CG_conv4_regrd.fits')
hdu_CG_UC  = fits.open('/srv/data/cgps-gmims/conv_regrid/UC_CG_conv4_regrd.fits')
hdu_CG_UD  = fits.open('/srv/data/cgps-gmims/conv_regrid/UD_CG_conv4_regrd.fits')

CG_QA = hdu_CG_QA[0].data
CG_QB = hdu_CG_QB[0].data
CG_QC = hdu_CG_QC[0].data
CG_QD = hdu_CG_QD[0].data

CG_UA = hdu_CG_UA[0].data
CG_UB = hdu_CG_UB[0].data
CG_UC = hdu_CG_UC[0].data
CG_UD = hdu_CG_UD[0].data

print(CG_UA.shape)

hdr_CG_QA = hdu_CG_QA[0].header
hdr_CG_QB = hdu_CG_QB[0].header
hdr_CG_QC = hdu_CG_QC[0].header
hdr_CG_QD = hdu_CG_QD[0].header

hdr_CG_UA = hdu_CG_UA[0].header
hdr_CG_UB = hdu_CG_UB[0].header
hdr_CG_UC = hdu_CG_UC[0].header
hdr_CG_UD = hdu_CG_UD[0].header

In [ ]:
data_list = [CG_QA, CG_QB, CG_QC, CG_QD, CG_UA, CG_UB, CG_UC, CG_UD]

unit_list = ['K   ','K   ','K   ','K   ','K   ','K   ','K   ','K   ']

hdr_list  = [hdr_CG_QA, hdr_CG_QB, hdr_CG_QC, hdr_CG_QD,
             hdr_CG_UA, hdr_CG_UB, hdr_CG_UC, hdr_CG_UD]

telescope_list = ['DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m',
                  'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m', 'DRAO ST + Galt 26-m']

outdir = '/srv/data/cgps-gmims/data_for_paper/'

outfile_list = [outdir+'combined_Q_chanA.fits', outdir+'combined_Q_chanB.fits',
                outdir+'combined_Q_chanC.fits', outdir+'combined_Q_chanD.fits',
                outdir+'combined_U_chanA.fits', outdir+'combined_U_chanB.fits',
                outdir+'combined_U_chanC.fits', outdir+'combined_U_chanD.fits']

comment1_list = ['    Combined CGPS & GMIMS-HBN Stokes Q convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes Q convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes Q convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes Q convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes U convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes U convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes U convolved to 3 arcmin',
                 '    Combined CGPS & GMIMS-HBN Stokes U convolved to 3 arcmin']

comment2_list = ['    Channel A, 1407 MHz',
                 '    Channel B, 1414 MHz',
                 '    Channel C, 1428 MHz',
                 '    Channel D, 1435 MHz',
                 '    Channel A, 1407 MHz',
                 '    Channel B, 1414 MHz',
                 '    Channel C, 1428 MHz',
                 '    Channel D, 1435 MHz']

comment3_list = [' ', ' ', ' ', ' ',' ', ' ', ' ', ' ']

In [ ]:
for i in range(0,len(data_list)):

    modify_header_and_save_file(hdr_list[i], data_list[i], telescope_list[i], unit_list[i],
                                comment1_list[i], comment2_list[i], comment3_list[i], outfile_list[i])

## RM file for referee with no r-value or p-value in extensions

In [ ]:
hdu_CG_RM = fits.open('/srv/data/cgps-gmims/data_for_paper/RM_CG_conv4_regrd.fits')

hdr_CG_RM = hdu_CG_RM[0].header

keep_hdul = fits.HDUList([hdu_CG_RM[0], hdu_CG_RM[1], hdu_CG_RM[4], hdu_CG_RM[5]])

print(keep_hdul[0].data.shape)
print(keep_hdul[1].data.shape)
print(keep_hdul[2].data.shape)
print(keep_hdul[3].data.shape)

In [ ]:
hdu_list = [keep_hdul]

telescope_list = ['DRAO ST + Galt 26-m']

unit_list = ['RAD/M^2', 'RAD', 'RAD/M^2', 'RAD']

outdir = '/srv/data/cgps-gmims/data_for_paper/'

outfile_list = [outdir+'combined_RM.fits']

comment1_list = ['    RM from combined CGPS & GMIMS-HBN Q & U files convolved to 3 arcmin']

comment2_list = ['    Extensions: 1 linear fit RM, 2 polarization angle y-intercept,']

comment3_list = ['    3 linear fit error, 4 PA y-intercept error']

In [ ]:
for i in range(0,len(hdu_list)):

    modify_hdu_and_save_file_v2(hdu_list[i], telescope_list[i], unit_list,
                                comment1_list[i], comment2_list[i], comment3_list[i], outfile_list[i])

In [ ]:
#this = fits.open('/srv/data/cgps-gmims/data_for_paper/combined_RM.fits')
this = fits.open('/srv/data/cgps-gmims/data_for_paper/combined_RM.fits')

print(this[0].data.shape)
plt.hist(this[1].data.flatten(),bins=101,range=(-6,6),alpha=0.5);
plt.hist(this[3].data.flatten(),bins=101,range=(-6,6),alpha=0.5);
plt.ylim(0,5e5)

print(repr(this[1].header))

In [ ]:
plt.imshow(this[0].data,vmin=-300,vmax=300,cmap='RdBu_r')